In [1]:
import kagglehub
path = kagglehub.dataset_download("nalisha/mall-customer-segmentation-dataset-for-behavioral")
print("Path to dataset files:", path)

100%|██████████| 1.55k/1.55k [00:00<00:00, 2.27MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/nalisha/mall-customer-segmentation-dataset-for-behavioral/versions/1


In [2]:
import pandas as pd

In [3]:
csv_path = path + '/Mall_Customers.csv'
df = pd.read_csv(csv_path)

df.info()
print('Информация о датасете:')
print(f'Количество строк:{df.shape[0]}')
print(f'Количество столбцов:{df.shape[1]}')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   CustomerID              200 non-null    int64 
 1   Gender                  200 non-null    object
 2   Age                     200 non-null    int64 
 3   Annual Income (k$)      200 non-null    int64 
 4   Spending Score (1-100)  200 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 7.9+ KB
Информация о датасете:
Количество строк:200
Количество столбцов:5


In [4]:
from google.colab import auth

auth.authenticate_user()

In [5]:
from google.cloud import bigquery

project_id = "my-project-savkinam"

client = bigquery.Client(project=project_id)

In [9]:
#BQ чувствителен к названием колонок, а у нас колонки с посторонними знаками
import re
df.columns = [re.sub(r'[^a-zA-Z0-9_]', '_', col) for col in df.columns]

In [10]:
dataset_id = "colab_clast"
table_name = "Mall_Customerst"

table_id = f"{project_id}.{dataset_id}.{table_name}"

#Загрузка в BQ (при повторном запуске не добавляем)
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

job = client.load_table_from_dataframe(
    df,
    table_id,
    job_config=job_config
)

job.result()

print("Таблица успешно загружена")

Таблица успешно загружена


In [11]:
#Проверка количества строк
query = f"""
SELECT COUNT(*) AS total_rows
FROM `{table_id}`
"""

client.query(query).to_dataframe()

,total_rows
0,200


In [16]:
# Пропуски
query = f"""
SELECT
    COUNTIF(CustomerID IS NULL) AS customer_id_null,
    COUNTIF(Gender IS NULL) AS gender_null,
    COUNTIF(Age IS NULL) AS age_null,
    COUNTIF(annual_income_k IS NULL) AS annual_income_null,
    COUNTIF(Spending_Score__1_100_ IS NULL) AS spending_score_null
FROM `{table_id}`
"""

client.query(query).to_dataframe()

,customer_id_null,gender_null,age_null,annual_income_null,spending_score_null
0,0,0,0,0,0


In [17]:
print('Распределение покупателей по полу')
query = f"""
SELECT
    Gender,
    COUNT(*) AS customers_count
FROM `{table_id}`
GROUP BY Gender
ORDER BY customers_count DESC;
"""
client.query(query).to_dataframe()

Распределение покупателей по полу


,Gender,customers_count
0,Female,112
1,Male,88


In [18]:
#Статистика по годовому доходу (k$)
print('Статистика по годовому доходу (k$)')
query = f"""
SELECT
    AVG(annual_income_k) AS avg_income,
    MIN(annual_income_k) AS min_income,
    MAX(annual_income_k) AS max_income
FROM `{table_id}`
"""
client.query(query).to_dataframe()

Статистика по годовому доходу (k$)


,avg_income,min_income,max_income
0,60.56,15,137


#Кластеризация

In [19]:
# Обучение K-Means
model_name = "my-project-savkinam.colab_clast.mall_kmeans_model"

query = f"""
CREATE OR REPLACE MODEL `{model_name}`
OPTIONS(
  model_type='KMEANS',
  num_clusters=5,
  kmeans_init_method='KMEANS++'
) AS
SELECT
    Age,
    annual_income_k,
    Spending_Score__1_100_
FROM `{table_id}`
"""
client.query(query).to_dataframe()
print("Модель успешно создана и обучена")

Модель успешно создана и обучена!


In [20]:
# Оценка качества кластеризации
query = f"""
SELECT * FROM ML.EVALUATE(MODEL `{model_name}`)
"""

df_metrics = client.query(query).to_dataframe()
df_metrics

,davies_bouldin_index,mean_squared_distance
0,0.981798,0.845945


In [21]:
# 3. Применение модели к данным
query = f"""
SELECT
    CENTROID_ID AS cluster_id,
    Age,
    annual_income_k,
    Spending_Score__1_100_
FROM ML.PREDICT(
    MODEL `{model_name}`,
    (
        SELECT
            Age,
            annual_income_k,
            Spending_Score__1_100_
        FROM `{table_id}`
    )
)
"""

df_predicted = client.query(query).to_dataframe()
df_predicted.head()

,cluster_id,Age,annual_income_k,Spending_Score__1_100_
0,3,37,78,1
1,3,34,103,23
2,3,46,98,15
3,4,59,54,47
4,2,23,54,52
